In [1]:
import requests
import xml.etree.ElementTree as ET


def parse_pubmed_article(article, query):
    # Title
    title_node = article.find(".//ArticleTitle")
    title = (
        "".join(title_node.itertext()).strip()
        if title_node is not None
        else None
    )

    # Abstract
    abstract_nodes = article.findall(".//Abstract/AbstractText")
    abstract_parts = []

    for node in abstract_nodes:
        text = "".join(node.itertext()).strip()
        label = node.attrib.get("Label")

        if label:
            abstract_parts.append(f"{label}: {text}")
        else:
            abstract_parts.append(text)

    abstract = " ".join(abstract_parts) if abstract_parts else None

    # Authors
    authors = []

    for author in article.findall(".//Author"):
        first = author.findtext("ForeName")
        last = author.findtext("LastName")

        if first or last:
            name = " ".join(
                part for part in [first, last]
                if part
            )
            authors.append(name)

    # Keywords
    keywords = [
        "".join(node.itertext()).strip()
        for node in article.findall(".//Keyword")
    ]

    # Year
    year_node = article.find(".//JournalIssue/PubDate/Year")
    year = year_node.text if year_node is not None else None

    # PMID
    pmid = article.findtext(".//PMID")

    # DOI
    doi = None

    for node in article.findall(".//ELocationID"):
        if node.attrib.get("EIdType") == "doi":
            doi = node.text
            break

    if doi is None:
        for node in article.findall(".//ArticleId"):
            if node.attrib.get("IdType") == "doi":
                doi = node.text
                break

    # URL
    url = (
        f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        if pmid
        else None
    )

    return {
        "query": query,
        "source": "PubMed",
        "source_id": pmid,
        "title": title,
        "authors": authors,
        "abstract": abstract,
        "keywords": keywords,
        "year": year,
        "doi": doi,
        "url": url
    }


def search_pubmed(query, max_results=5):
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    search_params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": max_results
    }

    search_response = requests.get(
        search_url,
        params=search_params,
        timeout=30
    )

    search_response.raise_for_status()

    search_data = search_response.json()
    pmids = search_data["esearchresult"]["idlist"]

    if not pmids:
        return []

    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    fetch_params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }

    fetch_response = requests.get(
        fetch_url,
        params=fetch_params,
        timeout=30
    )

    fetch_response.raise_for_status()

    root = ET.fromstring(fetch_response.content)
    articles = root.findall(".//PubmedArticle")

    return [
        parse_pubmed_article(article, query)
        for article in articles
    ]

In [2]:
import time
import requests
import xml.etree.ElementTree as ET


def parse_arxiv_entry(entry, query, ns):

    title = entry.findtext(
        "atom:title",
        default=None,
        namespaces=ns
    )

    abstract = entry.findtext(
        "atom:summary",
        default=None,
        namespaces=ns
    )

    authors = [
        author.findtext(
            "atom:name",
            default=None,
            namespaces=ns
        )
        for author in entry.findall("atom:author", ns)
    ]

    authors = [author for author in authors if author]

    published = entry.findtext(
        "atom:published",
        default=None,
        namespaces=ns
    )

    year = published[:4] if published else None

    doi = entry.findtext(
        "arxiv:doi",
        default=None,
        namespaces=ns
    )

    keywords = [
        node.attrib.get("term")
        for node in entry.findall("atom:category", ns)
        if node.attrib.get("term")
    ]

    source_id = entry.findtext(
        "atom:id",
        default=None,
        namespaces=ns
    )

    return {
        "query": query,
        "source": "arXiv",
        "source_id": source_id,
        "title": title.strip() if title else None,
        "authors": authors,
        "abstract": abstract.strip() if abstract else None,
        "keywords": keywords,
        "year": year,
        "doi": doi,
        "url": source_id
    }


def search_arxiv(query, max_results=5, wait_seconds=3, retries=3):

    arxiv_url = "https://export.arxiv.org/api/query"

    params = {
        "search_query": f'all:"{query}"',
        "start": 0,
        "max_results": max_results
    }

    headers = {
        "User-Agent": "ResearchMate/1.0"
    }

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    for attempt in range(retries):

        try:
            time.sleep(wait_seconds)

            response = requests.get(
                arxiv_url,
                params=params,
                headers=headers,
                timeout=60
            )

            if response.status_code == 429:
                print("Rate limited by arXiv. Waiting...")
                time.sleep(15)
                continue

            response.raise_for_status()

            root = ET.fromstring(response.content)

            entries = root.findall("atom:entry", ns)

            return [
                parse_arxiv_entry(entry, query, ns)
                for entry in entries
            ]

        except requests.exceptions.Timeout:
            print(
                f"arXiv request timed out "
                f"(attempt {attempt + 1}/{retries})"
            )

            time.sleep(10)

        except requests.exceptions.RequestException as e:
            print(
                f"arXiv request failed "
                f"(attempt {attempt + 1}/{retries}): {e}"
            )

            time.sleep(10)

    print("arXiv failed after all retries.")
    return []

In [3]:
def search_crossref(query, max_results=5):

    url = "https://api.crossref.org/works"

    params = {
        "query": query,
        "rows": max_results
    }

    headers = {
        "User-Agent": "ResearchMate/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    items = response.json()["message"]["items"]

    results = []

    for item in items:

        # Title
        titles = item.get("title", [])
        title = titles[0] if titles else None

        # Authors
        authors = []

        for author in item.get("author", []):
            first = author.get("given")
            last = author.get("family")

            name = " ".join(
                part for part in [first, last]
                if part
            )

            if name:
                authors.append(name)

        # Abstract
        abstract = item.get("abstract")

        # DOI
        doi = item.get("DOI")

        # Year
        year = None

        date_parts = (
            item.get("published-print", {})
            .get("date-parts", [])
        )

        if not date_parts:
            date_parts = (
                item.get("published-online", {})
                .get("date-parts", [])
            )

        if date_parts and date_parts[0]:
            year = str(date_parts[0][0])

        # URL
        paper_url = item.get("URL")

        # Crossref subjects
        keywords = item.get("subject", [])

        return_item = {
            "query": query,
            "source": "Crossref",
            "source_id": doi,
            "title": title,
            "authors": authors,
            "abstract": abstract,
            "keywords": keywords,
            "year": year,
            "doi": doi,
            "url": paper_url
        }

        results.append(return_item)

    return results

In [4]:
pubmed_results = search_pubmed(
    query="climate change agriculture",
    max_results=5
)

arxiv_results = search_arxiv(
    query="climate change agriculture",
    max_results=5
)

In [5]:
print("PubMed results:", len(pubmed_results))
print("arXiv results:", len(arxiv_results))

PubMed results: 5
arXiv results: 4


In [6]:
pubmed_results[0]

{'query': 'climate change agriculture',
 'source': 'PubMed',
 'source_id': '42874437',
 'title': 'Regulation of antioxidant defense and stress physiology by foliar chemical elements and osmoprotectants for enhanced fruit development in mandarin under heat and high irradiance.',
 'authors': ['Maha Mohammed Alharbi',
  'Khulud S Albalawi',
  'Makhdora Almuziny',
  'Mashael M Alotaibi',
  'Mashael Daghash Alqahtani',
  'Fahad Mohammed Alzuaibr',
  'Hussain Alqahtani',
  'Moodi S Alsubeie',
  'Sobhy M Khalifa',
  'Ahmed S Dawood',
  'Adel F Ahmed',
  'Mohamed A Shahda',
  'Mamdouh M A Awad-Allah',
  'Azza H Mohamed'],
 'abstract': "Citrus fruits are frequently affected by high temperatures and severe solar radiation, particularly under arid and semi-arid environments, which causes extensive fruit sunburn and leads to significant losses in both yield and commercial quality. Developing sustainable, low cost, and practical strategies to mitigate sunburn damage while maintaining optimal fruit 

In [7]:
arxiv_results[0]

{'query': 'climate change agriculture',
 'source': 'arXiv',
 'source_id': 'http://arxiv.org/abs/2501.12535v1',
 'title': 'How Does the Spatial Distribution of Pre-training Data Affect Geospatial Foundation Models?',
 'authors': ['Mirali Purohit',
  'Gedeon Muhawenayo',
  'Esther Rolf',
  'Hannah Kerner'],
 'abstract': 'Foundation models have made rapid advances in many domains including Earth observation, where Geospatial Foundation Models (GFMs) can help address global challenges such as climate change, agriculture, and disaster response. Previous work on GFMs focused on tailoring model architecture and pre-text tasks, and did not investigate the impact of pre-training data selection on model performance. However, recent works from other domains show that the pre-training data distribution is an important factor influencing the performance of the foundation models. With this motivation, our research explores how the geographic distribution of pre-training data affects the performance 

In [8]:
import pandas as pd

all_results = pubmed_results + arxiv_results

df = pd.DataFrame(all_results)

df

,query,source,source_id,title,authors,abstract,keywords,year,doi,url
0,climate change agriculture,PubMed,42874437,Regulation of antioxidant defense and stress p...,"[Maha Mohammed Alharbi, Khulud S Albalawi, Mak...",Citrus fruits are frequently affected by high ...,"[Climate change, Murcotte Tangarine, Potassium...",2026,10.1016/j.plaphy.2026.111845,https://pubmed.ncbi.nlm.nih.gov/42874437/
1,climate change agriculture,PubMed,42873755,Evolutionary constraint on a core winter prote...,"[Elad Oren, Jingjing Zhai, Travis E Rooney, Ru...","Grasses in the Panicoideae, Aristidoideae, Chl...",[],2026,10.1371/journal.pone.0358458,https://pubmed.ncbi.nlm.nih.gov/42873755/
2,climate change agriculture,PubMed,42873330,Host Compatibility of the Temnocephalid Dicera...,"[N Azri-Shah Norhan, Nor Asma Husna Yusoff, Fa...",BACKGROUND: The introduction of host-associate...,"[Crayfish, Ecosystem, Ectosymbiont, Invasion, ...",2026,10.1007/s11686-026-01415-w,https://pubmed.ncbi.nlm.nih.gov/42873330/
3,climate change agriculture,PubMed,42871476,Climate-driven shifts in lumpy skin disease ri...,"[Seyed Erfan Momenpour, Masoumeh Moghbel, Seye...",This study aimed to identify the environmental...,"[Climate change, Ensemble models, Lumpy skin d...",2026,10.1016/j.onehlt.2026.101602,https://pubmed.ncbi.nlm.nih.gov/42871476/
4,climate change agriculture,PubMed,42870707,NPF and sNPF can regulate lipid metabolism to ...,"[Ruichuan Duan, Xinru Xiao, Yewei Jia, Shuqi R...",BACKGROUND: Elevated atmospheric carbon dioxid...,"[climate change, fatty acid, neuropeptide, oxi...",2026,10.1002/ps.71367,https://pubmed.ncbi.nlm.nih.gov/42870707/
5,climate change agriculture,arXiv,http://arxiv.org/abs/2501.12535v1,How Does the Spatial Distribution of Pre-train...,"[Mirali Purohit, Gedeon Muhawenayo, Esther Rol...",Foundation models have made rapid advances in ...,"[cs.LG, cs.CV]",2025,NaN,http://arxiv.org/abs/2501.12535v1
6,climate change agriculture,arXiv,http://arxiv.org/abs/2402.09488v2,Intelligent Agricultural Greenhouse Control Sy...,"[Cangqing Wang, Jiangchuan Gong]",This study endeavors to conceptualize and exec...,"[eess.SY, cs.LG]",2024,NaN,http://arxiv.org/abs/2402.09488v2
7,climate change agriculture,arXiv,http://arxiv.org/abs/2201.10985v1,Jalisco's multiclass land cover analysis and c...,"[Alexander Quevedo, Abraham Sánchez, Raul Nanc...","The understanding of global climate change, ag...","[cs.CV, cs.AI]",2022,NaN,http://arxiv.org/abs/2201.10985v1
8,climate change agriculture,arXiv,http://arxiv.org/abs/2405.20656v5,Automatic Counting and Classification of Mosqu...,"[Javier Naranjo-Alcazar, Jordi Grau-Haro, Pedr...","Insect pest control poses a global challenge, ...",[cs.AI],2024,NaN,http://arxiv.org/abs/2405.20656v5


In [9]:
df.columns

Index(['query', 'source', 'source_id', 'title', 'authors', 'abstract',
       'keywords', 'year', 'doi', 'url'],
      dtype='str')

In [10]:
queries = [
    "machine learning healthcare",
    "climate change agriculture",
    "renewable energy storage",
    "quantum computing algorithms",
    "artificial intelligence education",
    "natural language processing misinformation",
    "computer vision medical imaging",
    "cybersecurity intrusion detection",
    "robotics human interaction",
    "large language models scientific research",
    "biodiversity conservation",
    "microplastics marine ecosystems",
    "air pollution human health",
    "antibiotic resistance",
    "cancer immunotherapy",
    "gene editing CRISPR",
    "mental health social media",
    "food security climate change",
    "urban sustainability",
    "electric vehicle batteries",
    "solar energy forecasting",
    "economic inequality",
    "remote sensing agriculture",
    "protein structure prediction",
    "gravitational waves astronomy"
]

In [11]:
import time

all_results = []

for i, query in enumerate(queries, start=1):

    print(f"\nQuery {i}/{len(queries)}: {query}")

    # PubMed
    try:
        pubmed_results = search_pubmed(
            query=query,
            max_results=5
        )

        all_results.extend(pubmed_results)

        print("PubMed:", len(pubmed_results))

    except Exception as e:
        print("PubMed failed:", e)

    # Small pause before arXiv
    time.sleep(2)

    # arXiv
    try:
        arxiv_results = search_arxiv(
            query=query,
            max_results=5,
            wait_seconds=3,
            retries=3
        )

        all_results.extend(arxiv_results)

        print("arXiv:", len(arxiv_results))

    except Exception as e:
        print("arXiv failed:", e)

    # Important pause before next query
    time.sleep(5)


Query 1/25: machine learning healthcare
PubMed: 5
arXiv: 5

Query 2/25: climate change agriculture
PubMed: 5
arXiv: 4

Query 3/25: renewable energy storage
PubMed: 5
arXiv: 5

Query 4/25: quantum computing algorithms
PubMed: 5
arXiv: 5

Query 5/25: artificial intelligence education
PubMed: 5
arXiv: 5

Query 6/25: natural language processing misinformation
PubMed: 5
arXiv: 0

Query 7/25: computer vision medical imaging
PubMed: 5
arXiv: 5

Query 8/25: cybersecurity intrusion detection
PubMed: 5
arXiv: 5

Query 9/25: robotics human interaction
PubMed: 5
arXiv: 5

Query 10/25: large language models scientific research
PubMed: 5
arXiv: 0

Query 11/25: biodiversity conservation
PubMed: 5
arXiv: 5

Query 12/25: microplastics marine ecosystems
PubMed: 5
arXiv: 0

Query 13/25: air pollution human health
PubMed: 5
arXiv: 0

Query 14/25: antibiotic resistance
PubMed: 5
arXiv: 5

Query 15/25: cancer immunotherapy
PubMed: 5
arXiv: 5

Query 16/25: gene editing CRISPR
PubMed: 5
arXiv: 0

Query 17/25

In [12]:
import pandas as pd

df_25 = pd.DataFrame(all_results)

df_25

,query,source,source_id,title,authors,abstract,keywords,year,doi,url
0,machine learning healthcare,PubMed,42874426,Machine learning-based prediction of hospital-...,"[Yonghao Li, Xiaoyi Luan, Yilin Li, Liru Chen,...",OBJECTIVE: This study aimed to investigate the...,"[GLIM criteria, Hospital-acquired malnutrition...",2026,10.1016/j.nut.2026.113413,https://pubmed.ncbi.nlm.nih.gov/42874426/
1,machine learning healthcare,PubMed,42873894,Innovative roadmaps for sustainable antimicrob...,"[Gianmarco Gasperini, Silvia Rossi Paccani, Ka...",Integrated strategies to combat antibiotic-res...,[],2026,10.1126/scitranslmed.aef8416,https://pubmed.ncbi.nlm.nih.gov/42873894/
2,machine learning healthcare,PubMed,42873721,Integrating artificial intelligence in oncolog...,"[Ashwaq Salami, Ahmed Sohaib]",BACKGROUND: Artificial intelligence has the po...,"[artificial intelligence, chemotherapy, oncolo...",2026,10.12968/ijpn.2024.0080,https://pubmed.ncbi.nlm.nih.gov/42873721/
3,machine learning healthcare,PubMed,42873153,Digital twins for personalized management of o...,"[Motahareh Soltani, Somayyeh Karami-Mohajeri, ...","INTRODUCTION: The opioid crisis, affecting ove...",[],2026,10.5055/jom.0987,https://pubmed.ncbi.nlm.nih.gov/42873153/
4,machine learning healthcare,PubMed,42872962,A machine learning framework for precision dos...,"[Xin-Fang Zhang, Jing Sun, Guo-Xiang Hao, Xiao...",The large pharmacokinetic variability of dapto...,"[daptomycin, individual treatment, machine lea...",2026,10.1128/aac.01089-26,https://pubmed.ncbi.nlm.nih.gov/42872962/
...,...,...,...,...,...,...,...,...,...,...
210,gravitational waves astronomy,arXiv,http://arxiv.org/abs/0903.0103v1,The Promise of Low-Frequency Gravitational Wav...,[T. A. Prince],This Astro2010 science white paper provides an...,"[astro-ph.CO, astro-ph.HE]",2009,NaN,http://arxiv.org/abs/0903.0103v1
211,gravitational waves astronomy,arXiv,http://arxiv.org/abs/1012.1109v1,The road to gravitational-wave astronomy,[N. Andersson],Gravitational-wave astronomy is an area of gre...,[astro-ph.CO],2010,10.1016/j.ppnp.2011.01.013,http://arxiv.org/abs/1012.1109v1
212,gravitational waves astronomy,arXiv,http://arxiv.org/abs/gr-qc/9911034v1,Gravitational wave astronomy,[Bernard F Schutz],The first decade of the new millenium should s...,[gr-qc],1999,10.1088/0264-9381/16/12A/307,http://arxiv.org/abs/gr-qc/9911034v1
213,gravitational waves astronomy,arXiv,http://arxiv.org/abs/1804.06308v1,Gravitational Wave Astronomy: Delivering on th...,[Bernard F Schutz],Now that LIGO and Virgo have begun to detect g...,"[astro-ph.IM, astro-ph.CO, gr-qc]",2018,10.1098/rsta.2017.0279,http://arxiv.org/abs/1804.06308v1


In [13]:
df_25["source"].value_counts()

source
PubMed    125
arXiv      90
Name: count, dtype: int64

In [14]:
comparison = (
    df_25
    .groupby(["query", "source"])
    .size()
    .unstack(fill_value=0)
)

comparison

source,PubMed,arXiv
query,,
air pollution human health,5,0
antibiotic resistance,5,5
artificial intelligence education,5,5
biodiversity conservation,5,5
cancer immunotherapy,5,5
climate change agriculture,5,4
computer vision medical imaging,5,5
cybersecurity intrusion detection,5,5
economic inequality,5,5


In [16]:
df_25.to_csv("researchmate_results.25.csv", index=False)